In [1]:
import mysql.connector
import pandas as pd
from dotenv import load_dotenv
from urllib.parse import quote_plus
from sqlalchemy import create_engine
import os
import sys

# Add project root to path so we can import from src if needed later
sys.path.append(os.path.abspath('..'))

# Load environment variables from .env file
load_dotenv(dotenv_path='../.env')

print('Imports successful')

Imports successful


In [2]:
def get_connection():
    """Raw MySQL connection for inserts."""
    return mysql.connector.connect(
        host=os.getenv('DB_HOST'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD'),
        database=os.getenv('DB_NAME')
    )



def get_engine():
    """SQLAlchemy engine for pandas read_sql."""
    password = quote_plus(os.getenv('DB_PASSWORD'))
    return create_engine(
        f"mysql+mysqlconnector://{os.getenv('DB_USER')}:{password}"
        f"@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}"
    )

# Test connection

try:
    conn = get_connection()
    print(f"Connected to MySQL database: {os.getenv('DB_NAME')}")
    conn.close()
except Exception as e:
    print(f"Connection failed: {e}")

Connected to MySQL database: gtavi_economy_map


In [3]:
GAME_PRICES = [
    # ── Rockstar Games ──────────────────────────────────────────────────────
    # GTA IV marked the start of the modern AAA pricing era at $59.99
    ('Grand Theft Auto IV',              'Rockstar Games', 2008, 'PS3/Xbox 360',    1, 59.99, None,  0),
    ('Red Dead Redemption',              'Rockstar Games', 2010, 'PS3/Xbox 360',    1, 59.99, None,  0),
    ('Grand Theft Auto V',               'Rockstar Games', 2013, 'PS3/Xbox 360',    1, 59.99, 79.99, 1),
    ('Grand Theft Auto V (PC)',          'Rockstar Games', 2015, 'PC',              2, 59.99, 79.99, 1),
    ('Red Dead Redemption 2',            'Rockstar Games', 2018, 'PS4/Xbox One',    2, 59.99, 99.99, 1),
    ('Red Dead Redemption 2 (PC)',       'Rockstar Games', 2019, 'PC',              2, 59.99, 99.99, 1),


    # ── Activision / Call of Duty ────────────────────────────────────────────
    # COD is the most consistent annual AAA franchise — ideal pricing benchmark
    # Vanguard excluded: underperformed commercially and critically,
    # making it an outlier that would skew the model

    ('Call of Duty 4: Modern Warfare',   'Activision',     2007, 'PS3/Xbox 360',    1, 59.99, None,  0),
    ('Call of Duty: Modern Warfare 2',   'Activision',     2009, 'PS3/Xbox 360',    1, 59.99, None,  0),
    ('Call of Duty: Black Ops',          'Activision',     2010, 'PS3/Xbox 360',    1, 59.99, None,  0),
    ('Call of Duty: Modern Warfare 3',   'Activision',     2011, 'PS3/Xbox 360',    1, 59.99, None,  0),
    ('Call of Duty: Black Ops II',       'Activision',     2012, 'PS3/Xbox 360',    1, 59.99, None,  0),
    ('Call of Duty: Ghosts',             'Activision',     2013, 'PS3/Xbox 360',    1, 59.99, 79.99, 1),
    ('Call of Duty: Advanced Warfare',   'Activision',     2014, 'PS4/Xbox One',    2, 59.99, 79.99, 1),
    ('Call of Duty: Black Ops III',      'Activision',     2015, 'PS4/Xbox One',    2, 59.99, 99.99, 1),
    ('Call of Duty: Infinite Warfare',   'Activision',     2016, 'PS4/Xbox One',    2, 59.99, 79.99, 1),
    ('Call of Duty: WWII',               'Activision',     2017, 'PS4/Xbox One',    2, 59.99, 99.99, 1),
    ('Call of Duty: Black Ops 4',        'Activision',     2018, 'PS4/Xbox One',    2, 59.99, 99.99, 1),
    ('Call of Duty: Modern Warfare',     'Activision',     2019, 'PS4/Xbox One',    2, 59.99, 99.99, 1),
    ('Call of Duty: Black Ops Cold War', 'Activision',     2020, 'PS5/Xbox Series', 3, 69.99, 99.99, 1),
    ('Call of Duty: Modern Warfare II',  'Activision',     2022, 'PS5/Xbox Series', 3, 69.99, 99.99, 1),
    ('Call of Duty: Modern Warfare III', 'Activision',     2023, 'PS5/Xbox Series', 3, 69.99, 99.99, 1),

    # ── EA ───────────────────────────────────────────────────────────────────
    # Mix of Battlefield (shooter) and FIFA/FC (sports) captures
    # EA's pricing across two very different game categories

    ('Battlefield 3',                    'EA',             2011, 'PS3/Xbox 360',    1, 59.99, None,  0),
    ('Battlefield 4',                    'EA',             2013, 'PS3/Xbox 360',    1, 59.99, 79.99, 1),
    ('FIFA 14',                          'EA',             2013, 'PS3/Xbox 360',    1, 59.99, None,  0),
    ('Battlefield 1',                    'EA',             2016, 'PS4/Xbox One',    2, 59.99, 79.99, 1),
    ('FIFA 18',                          'EA',             2017, 'PS4/Xbox One',    2, 59.99, 79.99, 1),
    ('Battlefield V',                    'EA',             2018, 'PS4/Xbox One',    2, 59.99, 79.99, 1),
    ('FIFA 22',                          'EA',             2021, 'PS5/Xbox Series', 3, 69.99, 99.99, 1),
    ('Battlefield 2042',                 'EA',             2021, 'PS5/Xbox Series', 3, 69.99, 99.99, 1),
    ('EA Sports FC 24',                  'EA',             2023, 'PS5/Xbox Series', 3, 69.99, 99.99, 1),

    # ── Sony First-Party ─────────────────────────────────────────────────────
    # Sony led the industry shift to $69.99 with PS5 launches in 2020
    # Including them captures that inflection point directly

    ('God of War',                       'Sony',           2018, 'PS4',             2, 59.99, None,  0),
    ('Marvel\'s Spider-Man',             'Sony',           2018, 'PS4',             2, 59.99, 79.99, 1),
    ('Horizon Zero Dawn',                'Sony',           2017, 'PS4',             2, 59.99, 79.99, 1),
    ('The Last of Us Part II',           'Sony',           2020, 'PS4',             2, 59.99, 79.99, 1),
    ('Demon\'s Souls',                   'Sony',           2020, 'PS5',             3, 69.99, None,  0),
    ('God of War Ragnarok',              'Sony',           2022, 'PS5',             3, 69.99, 79.99, 1),
    ('Marvel\'s Spider-Man 2',           'Sony',           2023, 'PS5',             3, 69.99, 79.99, 1),
    ('Horizon Forbidden West',           'Sony',           2022, 'PS5',             3, 69.99, 79.99, 1),

    # ── Nintendo ─────────────────────────────────────────────────────────────
    # Nintendo prices independently of competitive pressure due to
    # exclusive IP strength. Including them tests whether inflation
    # and platform generation drive pricing regardless of publisher strategy

    ('The Legend of Zelda: Breath of the Wild', 'Nintendo', 2017, 'Switch',        2, 59.99, None,  0),
    ('Super Mario Odyssey',              'Nintendo',       2017, 'Switch',          2, 59.99, None,  0),
    ('Pokemon Scarlet/Violet',           'Nintendo',       2022, 'Switch',          3, 59.99, None,  0),
    ('The Legend of Zelda: Tears of the Kingdom', 'Nintendo', 2023, 'Switch',      3, 69.99, None,  0),
]

print(f"Dataset contains {len(GAME_PRICES)} game records")
print(f"Years covered: {min(r[2] for r in GAME_PRICES)} to {max(r[2] for r in GAME_PRICES)}")
print(f"Publishers: {set(r[1] for r in GAME_PRICES)}")

Dataset contains 42 game records
Years covered: 2007 to 2023
Publishers: {'Sony', 'Nintendo', 'EA', 'Activision', 'Rockstar Games'}


In [4]:
columns = [
    'game_title', 'publisher', 'release_year', 'platform',
    'platform_generation', 'base_price_usd', 'premium_price_usd', 'had_premium_edition'
]

df_preview = pd.DataFrame(GAME_PRICES, columns=columns)

print(f"Shape: {df_preview.shape}")
print(f"\nRecords per publisher:")
print(df_preview['publisher'].value_counts().to_string())
print(f"\nRecords per platform generation:")
print(df_preview['platform_generation'].value_counts().sort_index().to_string())
print(f"\nPremium edition breakdown:")
print(df_preview['had_premium_edition'].value_counts().to_string())

df_preview

Shape: (42, 8)

Records per publisher:
publisher
Activision        15
EA                 9
Sony               8
Rockstar Games     6
Nintendo           4

Records per platform generation:
platform_generation
1    12
2    18
3    12

Premium edition breakdown:
had_premium_edition
1    27
0    15


,game_title,publisher,release_year,platform,platform_generation,base_price_usd,premium_price_usd,had_premium_edition
0,Grand Theft Auto IV,Rockstar Games,2008,PS3/Xbox 360,1,59.99,NaN,0
1,Red Dead Redemption,Rockstar Games,2010,PS3/Xbox 360,1,59.99,NaN,0
2,Grand Theft Auto V,Rockstar Games,2013,PS3/Xbox 360,1,59.99,79.99,1
3,Grand Theft Auto V (PC),Rockstar Games,2015,PC,2,59.99,79.99,1
4,Red Dead Redemption 2,Rockstar Games,2018,PS4/Xbox One,2,59.99,99.99,1
5,Red Dead Redemption 2 (PC),Rockstar Games,2019,PC,2,59.99,99.99,1
6,Call of Duty 4: Modern Warfare,Activision,2007,PS3/Xbox 360,1,59.99,NaN,0
7,Call of Duty: Modern Warfare 2,Activision,2009,PS3/Xbox 360,1,59.99,NaN,0
8,Call of Duty: Black Ops,Activision,2010,PS3/Xbox 360,1,59.99,NaN,0
9,Call of Duty: Modern Warfare 3,Activision,2011,PS3/Xbox 360,1,59.99,NaN,0


In [5]:
def load_game_prices(data):
    """
    Insert all game price records into MySQL.
    Clears existing data first so this cell is safe to rerun.
    """
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("DELETE FROM game_prices")

    insert_query = """
        INSERT INTO game_prices
            (game_title, publisher, release_year, platform,
             platform_generation, base_price_usd, premium_price_usd, had_premium_edition)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """

    cursor.executemany(insert_query, data)
    conn.commit()

    print(f"Loaded {cursor.rowcount} records into game_prices")
    cursor.close()
    conn.close()

load_game_prices(GAME_PRICES)

Loaded 42 records into game_prices


In [6]:
engine = get_engine()

df_verify = pd.read_sql("""
    SELECT
        publisher,
        game_title,
        release_year,
        platform_generation,
        base_price_usd,
        premium_price_usd,
        had_premium_edition

    FROM game_prices
    ORDER BY release_year, publisher
""", engine)

print(f"Total records in MySQL: {len(df_verify)}")
df_verify

Total records in MySQL: 42


,publisher,game_title,release_year,platform_generation,base_price_usd,premium_price_usd,had_premium_edition
0,Activision,Call of Duty 4: Modern Warfare,2007,1,59.99,NaN,0
1,Rockstar Games,Grand Theft Auto IV,2008,1,59.99,NaN,0
2,Activision,Call of Duty: Modern Warfare 2,2009,1,59.99,NaN,0
3,Activision,Call of Duty: Black Ops,2010,1,59.99,NaN,0
4,Rockstar Games,Red Dead Redemption,2010,1,59.99,NaN,0
5,Activision,Call of Duty: Modern Warfare 3,2011,1,59.99,NaN,0
6,EA,Battlefield 3,2011,1,59.99,NaN,0
7,Activision,Call of Duty: Black Ops II,2012,1,59.99,NaN,0
8,Activision,Call of Duty: Ghosts,2013,1,59.99,79.99,1
9,EA,Battlefield 4,2013,1,59.99,79.99,1


In [7]:
print("Null check:")
print(df_verify.isnull().sum().to_string())
print(f"\nPrice range check:")
print(f"  Base price min: ${df_verify['base_price_usd'].min()}")
print(f"  Base price max: ${df_verify['base_price_usd'].max()}")
print(f"  Premium price min: ${df_verify['premium_price_usd'].min()}")
print(f"  Premium price max: ${df_verify['premium_price_usd'].max()}")

print(f"\nYear range check:")
print(f"  Earliest: {df_verify['release_year'].min()}")
print(f"  Latest: {df_verify['release_year'].max()}")

print(f"\nDuplicate check:")
dupes = df_verify[df_verify.duplicated(subset=['game_title', 'release_year'], keep=False)]

if len(dupes) == 0:
    print("  No duplicates found")
else:
    print(f"  Warning: {len(dupes)} potential duplicates found")
    print(dupes[['game_title', 'release_year']].to_string())

Null check:
publisher               0
game_title              0
release_year            0
platform_generation     0
base_price_usd          0
premium_price_usd      15
had_premium_edition     0

Price range check:
  Base price min: $59.99
  Base price max: $69.99
  Premium price min: $79.99
  Premium price max: $99.99

Year range check:
  Earliest: 2007
  Latest: 2023

Duplicate check:
  No duplicates found


In [8]:
print("Notebook complete. Data validated and loaded into MySQL.")
print("Next step: run notebooks/02_clean_data.ipynb")

Notebook complete. Data validated and loaded into MySQL.
Next step: run notebooks/02_clean_data.ipynb
